# Example of evaluating electron repulsion integrals (ERIs) of Gausslet orbitals

In [1]:
# see the README for build and install instructions of this package
import gausslectronitegrate as gli

In [2]:
# Gausslet G10 coefficients
from gausslet_coefficients import get_gausslet_g10_coeffs
b_list, _ = get_gausslet_g10_coeffs()

In [3]:
# define a Cartesian grid by tuples of the form (istart, num points) in each coordinate direction
grid = [
    ( 0, 2),  # x
    ( 0, 1),  # y
    (-1, 3),  # z
]

In [4]:
# overall number of grid points
gli.grid_num_points(grid)

6

In [5]:
# truncation tolerance
# (values close to zero imply higher accuracy but longer computation time)
tol = 1e-16

In [6]:
# evaluate electron repulsion integrals (ERIs) for Gausslet orbitals centered at the grid points
eri, grid_trans = gli.compute_eri_gausslet_integrals(b_list, grid, tol)

In [7]:
# The calculation exploits translational invariance by subtracting the last orbital center:
# (i, j | k, l) -> (i - l, j - l | k - l, 0).
# Accordingly, the integral values are returned as a NumPy array of degree 3,
# where each dimension has length "number of translational grid points".
# This grid specification is the second return value:
grid_trans

[(-1, 3), (0, 1), (-2, 5)]

In [8]:
gli.grid_num_points(grid_trans)

15

In [9]:
eri.shape

(15, 15, 15)

In [10]:
# linear index of origin
iorigin_trans = gli.grid_point_to_linear_index(grid_trans, 0, 0, 0)
iorigin_trans

7

In [11]:
# show the ERI value for all Gausslets centered at the origin
eri[iorigin_trans, iorigin_trans, iorigin_trans]

1.6252009000830485

In [12]:
# evaluate the full electron repulsion integral (ERI) tensor of degree 4 (without exploiting translational invariance)
eri_full = gli.compute_full_eri_gausslet_integrals(b_list, grid, tol)
eri_full.shape

(6, 6, 6, 6)

In [13]:
# linear index of origin
iorigin = gli.grid_point_to_linear_index(grid, 0, 0, 0)

In [14]:
# show the ERI value for all Gausslets centered at the origin (should agree with above value)
eri_full[iorigin, iorigin, iorigin, iorigin]

1.6252009000830485

In [15]:
# check symmetries
import numpy as np
print(np.linalg.norm(eri_full.transpose((1, 0, 2, 3)) - eri_full))
print(np.linalg.norm(eri_full.transpose((0, 1, 3, 2)) - eri_full))
print(np.linalg.norm(eri_full.transpose((2, 3, 0, 1)) - eri_full))

0.0
0.0
0.0


In [16]:
# compute the individual overlap integral for all Gausslets centered at the origin (should agree with above value)
gli.compute_eri_gausslet_integral(b_list, ([0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]), tol)

1.6252009000830485

In [17]:
# evaluate the overlap integral for four other points, as an example
point_a = [ 1,  0,  0]
point_b = [ 0,  0, -1]
point_c = [ 0,  0,  1]
point_d = [ 0,  0,  0]
gli.compute_eri_gausslet_integral(b_list, (point_a, point_b, point_c, point_d), tol)

-0.0005864045861050048

In [18]:
# compare with value in 'eri_full' array
idx = [
    gli.grid_point_to_linear_index(grid, point_a[0], point_a[1], point_a[2]),
    gli.grid_point_to_linear_index(grid, point_b[0], point_b[1], point_b[2]),
    gli.grid_point_to_linear_index(grid, point_c[0], point_c[1], point_c[2]),
    gli.grid_point_to_linear_index(grid, point_d[0], point_d[1], point_d[2])]
eri_full[idx[0], idx[1], idx[2], idx[3]]

-0.0005864045861050048